# Amazon Laptop Scraping

In [1]:
import re
import requests
from bs4 import BeautifulSoup
import pandas as pd


In [2]:
url = "https://www.amazon.in/s"

In [3]:
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36",
    "Accept-Language": "en-IN,en;q=0.9"
}

In [4]:
# send reqest 
response=requests.get(url,headers=headers)
# check if the request was successful
response.status_code


200

In [5]:
# show the contain of the page
response.content

b'<!doctype html><html lang="en-in" class="a-no-js" data-19ax5a9jf="dingo"><!-- sp:feature:head-start -->\n<head><script>var aPageStart = (new Date()).getTime();</script><meta charset="utf-8"/>\n<!-- sp:end-feature:head-start -->\n<!-- sp:feature:csm:head-open-part1 -->\n\n<script type=\'text/javascript\'>var ue_t0=ue_t0||+new Date();</script>\n<!-- sp:end-feature:csm:head-open-part1 -->\n<!-- sp:feature:cs-optimization -->\n<meta http-equiv=\'x-dns-prefetch-control\' content=\'on\'>\n<link rel="preconnect" href="https://images-eu.ssl-images-amazon.com" crossorigin>\n<link rel="preconnect" href="https://m.media-amazon.com" crossorigin>\n<!-- sp:end-feature:cs-optimization -->\n<!-- sp:feature:csm:head-open-part2 -->\n<script type=\'text/javascript\'>\nwindow.ue_ihb = (window.ue_ihb || window.ueinit || 0) + 1;\nif (window.ue_ihb === 1) {\n\nvar ue_csm = window,\n    ue_hob = +new Date();\n(function(d){var e=d.ue=d.ue||{},f=Date.now||function(){return+new Date};e.d=function(b){return f()

In [6]:
# beautiful soup object
bs=BeautifulSoup(response.content,"html.parser")
print(bs)

<!DOCTYPE html>
<html class="a-no-js" data-19ax5a9jf="dingo" lang="en-in"><!-- sp:feature:head-start -->
<head><script>var aPageStart = (new Date()).getTime();</script><meta charset="utf-8"/>
<!-- sp:end-feature:head-start -->
<!-- sp:feature:csm:head-open-part1 -->
<script type="text/javascript">var ue_t0=ue_t0||+new Date();</script>
<!-- sp:end-feature:csm:head-open-part1 -->
<!-- sp:feature:cs-optimization -->
<meta content="on" http-equiv="x-dns-prefetch-control"/>
<link crossorigin="" href="https://images-eu.ssl-images-amazon.com" rel="preconnect"/>
<link crossorigin="" href="https://m.media-amazon.com" rel="preconnect"/>
<!-- sp:end-feature:cs-optimization -->
<!-- sp:feature:csm:head-open-part2 -->
<script type="text/javascript">
window.ue_ihb = (window.ue_ihb || window.ueinit || 0) + 1;
if (window.ue_ihb === 1) {

var ue_csm = window,
    ue_hob = +new Date();
(function(d){var e=d.ue=d.ue||{},f=Date.now||function(){return+new Date};e.d=function(b){return f()-(b?0:d.ue_t0)};e.st

In [7]:
# create  a empty list
list_data=[]

In [17]:
list_data = []

for page in range(1, 30):
    params = {"k": "laptops", "page": page}

    response = requests.get(url, headers=headers, params=params, timeout=30)
    response.raise_for_status()
    bs = BeautifulSoup(response.text, "html.parser")

    product_containers = bs.select('div[data-component-type="s-search-result"]')

    for product in product_containers:
        title_tag = product.select_one("h2 span")
        if not title_tag:
            continue

        title = title_tag.get_text(" ", strip=True)
        price_tag = product.select_one("span.a-price-whole")
        rating_tag = product.select_one("span.a-size-small.a-color-base")
        # Brand Ectraction
        match=re.search(r"^([A-Za-z]+)",title)
        brand=match.group(1) if match else "Unknown"

        #Step 6: Extract the RAM
        R_match = re.search(r"(\d+GB|RAM\s*(\d+GB)?|DDR\d?\s*(\d+GB)?|LPDDR\d?\s*(\d+GB)?)", title, re.IGNORECASE)
        ram = R_match.group(1) if R_match else "N/A"

        #step 7 : Extract the storage
        S_match = re.search(r"(\d+)\s*(GB|TB)\s*(?:SSD|Storage|HDD)", title, re.IGNORECASE)
        ssd_storage = f"{S_match.group(1)}{S_match.group(2)}" if S_match else "N/A"


         #Step 8: Extract the Color
        C_match = re.search(r"\b(Black|White|Silver|Gray|Grey|Red|Blue|Green|Yellow|Pink|Purple|Gold|Bronze|Rose Gold|Indigo|Glacier)\b", title, re.IGNORECASE)
        color = C_match.group(1) if C_match else "N/A"

         # Step 9: Extract the processor information
        # Step 7: Extract the processor (Intel, AMD, Apple M/A chip, Snapdragon, MediaTek, etc)
        processor = "N/A"
        # Try to match Apple M series (M1, M2, M3, M4, M5, etc.)
        P_match = re.search(r"Apple\s+M(\d+)", title, re.IGNORECASE)
        if P_match:
            processor = f"Apple M{P_match.group(1)}"
        else:
            # Try to match Apple A series (A18, A17, A16, etc.)
            P_match = re.search(r"Apple\s+A(\d+)", title, re.IGNORECASE)
            if P_match:
                processor = f"Apple A{P_match.group(1)}"
            else:
                # Try other processor keywords
                processor_keywords = ["Intel", "AMD", "Snapdragon", "MediaTek", "Celeron"]
                for keyword in processor_keywords:
                    if re.search(rf"\b{keyword}\b", title, re.IGNORECASE):
                        processor = keyword
                        break


        list_data.append({
            "title": title,
            "price": price_tag.get_text(strip=True) if price_tag else "N/A",
            "rating": rating_tag.get_text(strip=True) if rating_tag else "N/A",
            "brand": match.group(1) if match else "Unknown",
            "ram" : ram,
            "ssd_storage":ssd_storage,
            "color":color,
            "processor":processor
        })

In [21]:
# The extraction is performed in the previous cell.

In [22]:
# display the data item
for item in list_data:
    print(f"Title:    {item['title']}")
    print(f"Price:    {item['price']}")
    print(f"Rating:   {item['rating']}")
    print(f"Brand:    {item['brand']}")
    print(f"RAM:      {item['ram']}")
    print(f"Storage:  {item['ssd_storage']}")
    print(f"Color:    {item['color']}")
    print(f"Processor:{item['processor']}")
    print("--"*50)

    

Title:    HP 15 (i5 14th Gen), Intel Core 5, 16GB RAM (Upgradeable), 512GB SSD, FHD, Anti-Glare, 15.6''/39.6cm, Win11, M365 Basic(1yr), Office24, Silver,1.59kg, fd0682tu, FHD Camera w/Shutter, Backlit Laptop
Price:    75,990
Rating:   3.6
Brand:    HP
RAM:      16GB
Storage:  512GB
Color:    Silver
Processor:Intel
----------------------------------------------------------------------------------------------------
Title:    HP Omnibook 3, Snapdragon X Processor 45 Tops (16GB LPDDR5x,512GB SSD) 2K WUXGA, 14''/35.6cm, Win 11, M365*Office 24,Silver,1.42kg, hz0026QU/ hz0024QU, Lighter mini Charger, FHD IR Camera, AI Laptop
Price:    69,990
Rating:   3.2
Brand:    HP
RAM:      16GB
Storage:  512GB
Color:    Silver
Processor:Snapdragon
----------------------------------------------------------------------------------------------------
Title:    Lenovo V15 G4 AMD Athlon Silver 7120U Laptop 8GB LPDDR5 Ram, 512 GB SSD PCIe, Windows 11 Lifetime Validity,15.6" FHD Screen, AMD Radeon 610M, Silver, 

In [23]:
df=pd.DataFrame(list_data)
df



,title,price,rating,brand,ram,ssd_storage,color,processor
0,"HP 15 (i5 14th Gen), Intel Core 5, 16GB RAM (U...","75,990",3.6,HP,16GB,512GB,Silver,Intel
1,"HP Omnibook 3, Snapdragon X Processor 45 Tops ...","69,990",3.2,HP,16GB,512GB,Silver,Snapdragon
2,Lenovo V15 G4 AMD Athlon Silver 7120U Laptop 8...,"44,999",4.0,Lenovo,8GB,512GB,Silver,AMD
3,"Dell 15, Intel Core 13th Gen i5-1334U, FHD, 15...","69,990",3.8,Dell,16GB,512GB,Grey,Intel
4,Apple 2026 MacBook Neo 13″ Laptop with A18 Pro...,"73,990",4.8,Apple,8GB,256GB,Indigo,N/A
...,...,...,...,...,...,...,...,...
545,Apple 2026 MacBook Air 15″ Laptop with M5 chip...,"2,00,490",4.7,Apple,16GB,1TB,N/A,N/A
546,Apple 2026 MacBook Neo 13″ Laptop with A18 Pro...,"73,990",4.8,Apple,8GB,256GB,Indigo,N/A
547,Apple 2026 MacBook Neo 13″ Laptop with A18 Pro...,"73,990",4.7,Apple,8GB,256GB,Silver,N/A
548,Apple 2026 MacBook Pro Laptop with M5 Max chip...,"5,89,990",4.3,Apple,48GB,2TB,Black,N/A


In [24]:
df.to_csv("Amazon_laptops.csv",index=False)